In [2]:
proximidade <- function(qt, dt, janela = 3) {        # qt: tokens da consulta; dt: tokens do documento
  s <- 0                                             # o escore comeca em zero
  for (i in seq_along(qt)) for (j in seq_along(qt))  # todos os pares de posicoes da consulta...
    if (i < j && qt[i] != qt[j]) {                   # ...com i antes de j e termos diferentes
      pos_i <- which(dt == qt[i])                    # onde o termo i aparece no documento
      pos_j <- which(dt == qt[j])                    # onde o termo j aparece
      if (length(pos_i) && length(pos_j)) {          # os dois aparecem?
        d <- outer(pos_j, pos_i, "-")                # todas as distancias (j menos i)
        d <- d[d > 0 & d <= janela]                  # so j DEPOIS de i, e dentro da janela
        if (length(d)) s <- s + 1 / min(d)           # soma 1 / a menor distancia
      }
    }
  s                                                  # devolve o escore
}
reordenar <- function(rk, k, sc) {               # rk: ranking do estagio 1; sc: escores do estagio 2
  k <- min(k, length(rk))                        # protecao: k nao passa do numero de documentos
  top <- names(rk)[1:k]                          # a cabeca: os k primeiros
  nova <- top[order(-sc[top], seq_along(top))]   # pelo escore, decrescente; empate: ordem do estagio 1
  c(nova, names(rk)[-(1:k)])                     # cabeca reordenada + cauda intacta (NOMES)
}
proximidade(c("a", "b"), c("a", "x", "b"))                         # teste: "b" 2 depois de "a" -> 0.5
reordenar(c(a = 3, b = 2, c = 1), 2, c(a = 0.1, b = 0.9, c = 5))   # teste: tem que dar "b" "a" "c"


[1] 0.5

[1] "b" "a" "c"

In [3]:
source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor08.R")


In [4]:
docs <- docs_aula()          # os 8 documentos
cfg  <- cfg_aula()           # decisoes canonicas (janela 3)
ix   <- montar(docs, cfg)    # o indice

In [5]:
rb <- ranking_bm25("modelo de recuperacao", ix, cfg)        # estagio 1
qt <- preparar("modelo de recuperacao", cfg)                # modelo, de, recuperacao
px <- sapply(ix$tokens, function(dt) proximidade(qt, dt))   # estagio 2: d2 1, d3 0,5
g  <- gabarito_aula()                                       # os graus
curva <- t(sapply(1:8, function(k) metricas(reordenar(rb, k, px), g, 2)))   # uma linha por k
rownames(curva) <- paste0("k", 1:8)                         # nomeia as linhas
round(curva[, c("AP", "nDCG_grad")], 3)                     # qualidade em cada k

,AP,nDCG_grad
k1,0.833,0.951
k2,0.833,0.951
k3,1.000,0.982
k4,1.000,0.982
k5,1.000,0.982
k6,1.000,0.982
k7,1.000,0.982
k8,1.000,0.982


In [6]:
rerank_prox <- function(rk, qt, tokens, k, janela = 3) {   # estagio 2 honesto
  if (k == 0) return(names(rk))                            # k = 0: nao reordena
  top <- head(names(rk), k)                                # os candidatos
  sc  <- sapply(tokens[top], function(dt) proximidade(qt, dt, janela))   # so eles sao lidos
  reordenar(rk, k, sc)                                     # cabeca + cauda
}
rerank_prox(rb, qt, ix$tokens, 3)                          # igual a reordenar(rb, 3, px)

[1] "d2" "d3" "d1" "d4" "d8" "d6" "d5" "d7"

In [7]:
tempo <- sapply(0:8, function(k)                                   # para cada k...
  system.time(for (i in 1:5000) rerank_prox(rb, qt, ix$tokens, k))[["elapsed"]])   # ...5000 vezes
names(tempo) <- paste0("k", 0:8)                                   # nomeia
tempo                                                              # segundos por 5000 consultas

k0    k1    k2    k3    k4    k5    k6    k7    k8 
0.014 0.551 0.733 0.953 1.092 1.245 1.337 1.566 2.790

In [8]:
r <- relevancia(rb, g, 2)                                     # relevantes na ordem do BM25
sapply(1:8, function(k) recall_k(r$rel, k, r$R))              # R@k: o teto
rc <- ranking_cosseno("modelo de recuperacao", ix, cfg)       # outro estagio 1 (Aula 02)
names(rc)                                                     # onde ficou o d2?
round(sapply(1:8, function(k) metricas(reordenar(rc, k, px), g, 2)[["AP"]]), 3)   # a curva

[1] 0.5 0.5 1.0 1.0 1.0 1.0 1.0 1.0

[1] "d1" "d3" "d4" "d2" "d6" "d8" "d5" "d7"

[1] 0.50 0.75 0.75 1.00 1.00 1.00 1.00 1.00

In [12]:

REPO <- "https://raw.githubusercontent.com/andrecesar00/predicarga_andre_diego_kaykhy/main/"  # repositorio do grupo
source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor08.R")  # funcoes base
cfg <- tryCatch({                                              # configuracoes do grupo
  download.file(paste0(REPO, "estrutura/codigo/config.R"), "config.R", quiet = TRUE)
  source("config.R"); cfg
}, error = function(e) {
  cfg_aula()
})
cp     <- read.csv(paste0(REPO, "estrutura/banco-de-dados/corpus.csv"), stringsAsFactors = FALSE, fileEncoding = "UTF-8")  # corpus
docs   <- setNames(cp$texto, cp$id)                            # textos indexados por id
origem <- readRDS(gzcon(url(paste0(REPO, "estrutura/banco-de-dados/origem.rds"))))  # metadados de origem
ix     <- montar(docs, cfg)                                    # indice invertido do grupo

arqs <- c("qrels_Andre_2026-10-25.csv")                        # arquivo de julgamentos
for (a in arqs) download.file(paste0(REPO, "estrutura/banco-de-dados/", a), a)  # baixa o qrels
qrels <- ler_qrels(arqs, juiz = cfg$juizes)                    # carrega os graus de relevancia
nec   <- read.csv(paste0(REPO, "estrutura/banco-de-dados/necessidades.csv"), stringsAsFactors = FALSE, fileEncoding = "UTF-8-BOM")  # consultas
consultas <- setNames(nec$texto_consulta, nec$consulta)        # vetor nomeado de consultas
R_q  <- sapply(qrels, function(g) sum(g >= cfg$limiar))        # total de relevantes por consulta
usar <- names(qrels)[R_q > 0]                                  # consultas validas (com relevantes)


In [13]:
# sem a decisao da Aula 07 ou da 08 no config.R: os valores padrao do curso
if (is.null(cfg$modelo)) cfg$modelo <- "bm25"   # estagio 1 = BM25 (quem nao fez a Aula 07)
if (is.null(cfg$lsa_k))  cfg$lsa_k  <- 2        # so usado com modelo "lsa" ou hibrido
if (is.null(cfg$rrf_c))  cfg$rrf_c  <- 60       # so usado com modelo "hibrido"
if (is.null(cfg$janela)) cfg$janela <- 3        # janela da proximidade (Aula 08)
estagio1 <- function(texto, ix, cfg, m) {                       # o motor do grupo (Parte D)
  rb <- ranking_bm25(texto, ix, cfg)                            # BM25 com k1 e b do grupo
  if (cfg$modelo == "bm25") return(rb)                          # "bm25": pronto
  rl <- ranking_lsa(texto, ix, cfg, m)                          # LSA da Aula 07
  if (cfg$modelo == "lsa") rl else rrf(rb, rl, c = cfg$rrf_c)   # "lsa" ou hibrido
}
m <- if (cfg$modelo == "bm25") NULL else lsa(ix, cfg$lsa_k)     # LSA so se usado

dev   <- intersect(usar, nec$consulta[nec$conjunto == "dev"])          # aqui se escolhe
teste <- intersect(usar, nec$consulta[nec$conjunto == "teste"])        # so no fim
rks <- lapply(consultas[usar], function(tx) estagio1(tx, ix, cfg, m))  # estagio 1, uma vez
qts <- lapply(consultas[usar], function(tx) preparar(tx, cfg))         # tokens do grupo
seg1 <- system.time(for (q in dev) estagio1(consultas[[q]], ix, cfg, m))[["elapsed"]]
1000 * seg1 / length(dev)                                              # T1: ms por consulta

[1] 1.333333

In [14]:
med <- function(q, k, rks, qts, qrels, tokens, w, limiar)      # AP e nDCG de UMA consulta
  metricas(rerank_prox(rks[[q]], qts[[q]], tokens, k, w), qrels[[q]], limiar)[c("AP", "nDCG_grad")]
w  <- 3                                                        # a janela do grupo (Parte D)
ks <- c(0, 1, 2, 3, 5, 10, 20, 50, 100)                        # de 0 ate N
ks <- ks[ks <= ix$N]                                           # k nao passa de N

tab <- t(sapply(ks, function(k) {                              # uma linha por k
  seg <- system.time(for (i in 1:20) for (q in dev)            # 20 voltas no dev...
    rerank_prox(rks[[q]], qts[[q]], ix$tokens, k, w))[["elapsed"]]   # ...so o estagio 2
  mt <- rowMeans(sapply(dev, function(q) med(q, k, rks, qts, qrels, ix$tokens, w, cfg$limiar)))
  c(k = k, MAP = mt[["AP"]], nDCG = mt[["nDCG_grad"]], ms = 1000 * seg / (20 * length(dev)))   # ms por consulta
}))
round(tab, 3)

k,MAP,nDCG,ms
0,0.066,0.268,0.000
1,0.066,0.268,0.133
2,0.066,0.268,0.183
3,0.066,0.268,0.233
5,0.066,0.268,0.367
10,0.066,0.268,0.550
20,0.066,0.268,0.967
50,0.066,0.268,2.167


In [15]:
lapply(qrels[dev], function(g) names(g)[g >= cfg$limiar])   # IDs relevantes de cada consulta no dev

$q01
[1] "d43"

$q02
[1] "d59" "d57" "d15" "d8"  "d26"

$q03
[1] "d60" "d32"

In [21]:
ap0 <- sapply(dev, function(q) med(q, 0, rks, qts, qrels, ix$tokens, w, cfg$limiar)[["AP"]])   # estagio 1
ap1 <- sapply(dev, function(q) med(q, 5, rks, qts, qrels, ix$tokens, w, cfg$limiar)[["AP"]])   # k = 5 (troquem)
dev[ap1 > ap0]; dev[ap1 < ap0]                                 # ajudou; atrapalhou

character(0)

character(0)

In [22]:
k_final <- 0                                                   # escolha justificada no dev

seg_teste <- system.time(for (i in 1:20) for (q in teste)
  rerank_prox(rks[[q]], qts[[q]], ix$tokens, k_final, w))[["elapsed"]]

mt_teste <- rowMeans(sapply(teste, function(q)
  med(q, k_final, rks, qts, qrels, ix$tokens, w, cfg$limiar)))

c(k = k_final,
  MAP_teste = round(mt_teste[["AP"]], 3),
  nDCG_teste = round(mt_teste[["nDCG_grad"]], 3),
  ms_estagio2 = round(1000 * seg_teste / (20 * length(teste)), 3))

k   MAP_teste  nDCG_teste ms_estagio2 
      0.000       0.092       0.342       0.100

In [23]:
diag_dev <- do.call(rbind, lapply(dev, function(q) {
  qt     <- qts[[q]]                                         # termos uteis apos preparar()
  n_top  <- min(50, length(rks[[q]]))                        # limite seguro ate N
  top50  <- names(rks[[q]])[seq_len(n_top)]                  # top-50 do estagio 1

  # escore de proximidade de cada documento no top-50
  sc_prox   <- sapply(top50, function(d) proximidade(qt, ix$tokens[[d]], w))
  rk_depois <- rerank_prox(rks[[q]], qt, ix$tokens, n_top, w)[seq_len(n_top)]

  data.frame(
    consulta     = q,
    texto        = consultas[[q]],
    n_termos     = length(qt),                               # < 2 => nao existe par de termos
    docs_com_par = sum(sc_prox > 0),                         # docs com par dentro da janela w = 3
    max_prox     = round(max(sc_prox), 3),                   # maior escore de proximidade obtido
    mudou_ordem  = !identical(top50, rk_depois),             # TRUE se alguma posicao trocou
    stringsAsFactors = FALSE
  )
}))

diag_dev
qts[dev]                                                     # lista os tokens uteis de cada consulta

consulta,texto,n_termos,docs_com_par,max_prox,mudou_ordem
<chr>,<chr>,<int>,<int>,<dbl>,<lgl>
q01,monumentos historicos centro,3,0,0,FALSE
q02,fundacao vila martim afonso,4,21,1,TRUE
q03,ponte pensil arquitetura,3,2,1,FALSE


$q01
[1] "monumentos" "historicos" "centro"    

$q02
[1] "fundacao" "vila"     "martim"   "afonso"  

$q03
[1] "ponte"       "pensil"      "arquitetura"

In [24]:
# --- Estagio 1: Recuperacao base ---
if (is.null(cfg$modelo)) cfg$modelo <- "bm25"   # modelo escolhido para o estagio 1

# --- Estagio 2: Reordenamento por proximidade (Aula 08) ---
# Decisao baseada no dev (MAP = 0.066, nDCG = 0.268 estaveis para k = 0..50; T1 ~ 1.33 ms)
# Validacao unica no teste com k = 0: MAP = 0.092, nDCG = 0.342
cfg$rerank_k <- 0                               # k = 0 desativa o reordenamento (sem ganho no dev)
cfg$janela   <- 3                               # tamanho da janela de proximidade avaliada (w = 3)

In [25]:
buscar <- function(texto, ix, cfg, m = NULL) {
  rk <- estagio1(texto, ix, cfg, m)             # gera o ranking inicial (BM25)

  # Pula o estagio 2 quando rerank_k == 0, evitando o overhead de chamada (c0)
  if (!is.null(cfg$rerank_k) && cfg$rerank_k > 0) {
    qt <- preparar(texto, cfg)
    rk <- rerank_prox(rk, qt, ix$tokens, k = cfg$rerank_k, janela = cfg$janela)
  }

  rk
}